In [1]:
import sys
print(sys.version)
print("Day 2 - OOP")

3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
Day 2 - OOP


In [4]:
# Bad Way - passing raw dicts everywhere
doc1 = {"id": "001", "text": "RAG is powerful", "source": "paper.pdf"}
doc2 = {"id": "002", "text": "Vector databases store embeddings"}

# Problems:
# 1. doc2 has no "Source" key - keyError waiting to happen
# 2. No validation - anyone can add garbage keys
# 3. No methods - logic scattered everywhere
try:
    print(doc2["source"])# keyError!
except KeyError as e:
    print(f"KeyError : {e} - this why we need proper classes")

# No way to enforce structure, validate data, or add behaviour
print("\n We need a proper document class.")

KeyError : 'source' - this why we need proper classes

 We need a proper document class.


In [ ]:
from typing import Dict, List, Optional
from datetime import datetime
import hashlib

class Document:
    """
    Core document class for the RAG pipeline.
    Every piece of text flowing through the system
    will be wrapped in this class.
    """
    def __init__(
            self,
            text: str,
            source: str,
            doc_id: Optional[str] = None,
            metadata: Optional[Dict]= None
    ):
        # Validate inputs immediately
        if not text or not text.strip():
            raise ValueError("Document text cannot be empty")
        if not source:
            raise ValueError("Document source cannot be empty")
        
        self.text = text.strip()
        self.source= source
        self.metadata = metadata or {}
        self.created_at = datetime.now()
        self.embedding: Optional[List[float]]= None
        self.chunk_index: int=0

        # Auto_generate Id from content if not provided
        if doc_id:
            self.doc_id = doc_id
        else:
            self.doc_id = self._generate_id()
        
    def _generate_id(self)-> str :
        """Generate a unique id based on text content"""
        content = f"{self.source}_{self.text[:50]}"
        return hashlib.md5(content.encode()).hexdigest()[:12]
    
    def add_metadata(self, key: str, value: str)-> None:
        """Add a metadata foeld to the document"""
        self.metadata[key]= value

    def has_embedding(self) -> bool:
        """check if document has been embedded"""
        return self.embedding is not None
    
    def word_count(self)-> int:
        """Returns number of words in the document"""
        return len(self.text.split())

    def preview(self,chars: int = 50)-> str:
        """Return a short preview of the document text"""
        if len(self.text)<=chars:
            return self.text
        return self.text[:chars] + "..."

print("=== Creating Documents ===")
doc1 = Document(
    text = "RAG combines retrieval and generation for better answers",
    source = "research.pdf",
    metadata = {"page":"1", "author": "Jay" }
)

doc2= Document(
    text="Vector databases store and retrieve embeddings efficiently",
    source = "research.pdf"
)

print(f" Doc1 ID: {doc1.doc_id}")
print(f" Doc2 ID: {doc2.doc_id}")
print(f"Doc1 preview: {doc1.preview()}")
print(f"Doc1 word count: {doc1.word_count()}")
print(f"Doc1 has embedding: {doc1.has_embedding()}")
print(f"Doc1 metadata: {doc1.metadata}")

# Add metadata
doc1.add_metadata("language", "english")
print(f" Doc1 upadated metadata: {doc1.metadata}")

#Test validation
print("\n=== Testing Validation===")
try:
    bad_doc = Document(text="", source ="file.pdf")
except ValueError as e:
    print(f"caught: {e}")

try:
    bad_doc=Document(text="some text", source="")
except ValueError as e:
    print(f"Caught: {e}")

=== Craeting Documents ===
 Doc1 ID: c912f8b681d7
 Doc2 ID: f29224170e20
Doc1 preview: RAG combines retrieval and generation for better a...
Doc1 word count: 8
Doc1 has embedding: False
Doc1 metadata: {'page': '1', 'author': 'Jay'}
 Doc1 upadated metadata: {'page': '1', 'author': 'Jay', 'language': 'english'}

=== Testing Validation===
caught: Document text cannot be empty
Caught: Document source cannot be empty


In [ ]:
class Document:
    """
    Core document class for the RAG pipeline.
    Every piece of text flowing through the system
    will be wrapped in this class.
    """
    def __init__(
            self,
            text: str,
            source: str,
            doc_id: Optional[str] = None,
            metadata: Optional[Dict]= None
    ):
        # Validate inputs immediately
        if not text or not text.strip():
            raise ValueError("Document text cannot be empty")
        if not source:
            raise ValueError("Document source cannot be empty")
        
        self.text = text.strip()
        self.source= source
        self.metadata = metadata or {}
        self.created_at = datetime.now()
        self.embedding: Optional[List[float]]= None
        self.chunk_index: int=0

        # Auto_generate Id from content if not provided
        if doc_id:
            self.doc_id = doc_id
        else:
            self.doc_id = self._generate_id()
        
    def _generate_id(self)-> str :
        """Generate a unique id based on text content"""
        content = f"{self.source}_{self.text[:50]}"
        return hashlib.md5(content.encode()).hexdigest()[:12]
    
    def add_metadata(self, key: str, value: str)-> None:
        """Add a metadata field to the document"""
        self.metadata[key]= value

    def has_embedding(self) -> bool:
        """check if document has been embedded"""
        return self.embedding is not None
    
    def word_count(self)-> int:
        """Returns number of words in the document"""
        return len(self.text.split())

    def preview(self,chars: int = 50)-> str:
        """Return a short preview of the document text"""
        if len(self.text)<=chars:
            return self.text
        return self.text[:chars] + "..."

    # ----- Dunder Methods-----------

    def __str__(self) -> str:
        """Human readable - used by print()"""
        return f"Document(id={self.doc_id}, source={self.source}, words={self.word_count()})"

    def __repr__(self)-> str:
        """Developer readable - used in debugger and REPL"""
        return f"Document(doc_id='{self.doc_id}', source='{self.source}')"

    def __len__(self) -> int:
        """len(doc) returns word count"""
        return self.word_count()

    def __eq__(self, other) -> bool:
        """two documents are equal if they have the same ID"""
        if not isinstance(other, Document):
            return False
        return self.doc_id == other.doc_id

    def __lt__ (self, other) -> bool:
        """Compare documents by word count - enables sorting"""
        return self.word_count()< other.word_count()

    def __contains__(self, keyword:str)-> bool:
        """'Keyword' in document - searches for document text"""
        return keyword.lower() in self.text.lower()

doc1 = Document("RAG combines retrieval and generation", "paper.pdf")
doc2 = Document("Vector databases store embeddings eficiently for fast retrieval", "paper.pdf")
doc3 = Document("RAG combines retrieval and generation", "paper.pdf")

print("=== __str__ ===")
print(doc1) # calls __str__

print("\n=== __repr__ ===")
print(repr(doc1)) # calls __repr__

print("\n === __len__ ===")
print(f"doc1 length: {len(doc1)} words")
print(f"doc2 length: {len(doc2)} words")

print("\n === __eq__ ===")
print(f"doc1 == doc2: {doc1 == doc2}")
print(f"doc1 == doc3: {doc1==doc3}")

print("\n=== __lt__ -sorting ===")
docs = [doc2, doc1]
sorted_docs = sorted(docs)
for d in sorted_docs:
    print(f"{len(d)} words - {d.preview()}")

print("\n=== __contains__ ===")
print(f"RAG in doc1: {'RAG' in doc1}")
print(f"'vector' in doc2: {'vector' in doc1}")
print(f"'vector' in doc2: {'vector' in  doc2}")

=== __str__ ===
Document(id=b49dd2942685, source=paper.pdf, words=5)

=== __repr__ ===
Document(doc_id='b49dd2942685', source='paper.pdf')

 === __len__ ===
doc1 length: 5 words
doc2 length: 8 words

 === __eq__ ===
doc1 == doc2: False
doc1 == doc3: True

=== __lt__ -sorting ===
5 words - RAG combines retrieval and generation
8 words - Vector databases store embeddings eficiently for f...

=== __contains__ ===
RAG in doc1: True
'vector' in doc2: False
'vector' in doc2: True


In [23]:
from typing import Iterator

class DocumentCollection:
    """
    Manages a collection of documents for a single user/tenant,
    This is what sits between your pdf loader and chromadb.
    """

    def __init__(self, user_id: str, collection_name: str):
        self.user_id = user_id
        self.collection_name = collection_name
        self._documents: Dict[str, Document]= {} # id -> document

    def add(self, doc: Document)-> None:
        """Add document to the collection"""
        if not isinstance(doc, Document):
            raise TypeError("Only document objects can be added")
        if doc.doc_id in self._documents:
            print(f"[WARN] Document {doc.doc_id} already exists - skipping")
            return 
        self._documents[doc.doc_id] = doc
        print(f"[ADDED] {doc.doc_id} - {doc.preview()}")

    def get(self, doc_id:str) -> Optional[Document]:
        """Get a document by ID"""
        return self._documents.get(doc_id)
    
    def remove(self, doc_id:str)-> bool:
        """Remove a document by ID"""
        if doc_id in self._documents:
            del self._documents[doc_id]
            print(f"[REMOVED] {doc_id}")
            return True
        print(f"[WARN] Document {doc_id} not found")
        return False

    def search(self, keyword:str) -> List[Document]:
        """Search documents by keyword"""
        return [doc for doc in self._documents.values() if keyword in doc]   
    
    def unembedded(self) -> List[Document]:
        """Return all documents that haven't beenn embeddedd yet"""
        return [doc for doc in self._documents.values() if not doc.has_embedding()]
    
    def embedded(self)-> List[Document]:
        """Returns all the documents that have been embedded"""
        return [doc for doc in self._documents.values() if doc.has_embedding()]
    
    def stats(self)-> Dict:
        """Return collection statistics"""
        total = len(self._documents)
        embedded_count = len(self.embedded())
        return{
            "user_id": self.user_id,
            "collection": self.collection_name,
            "total_documents": total,
            "embedded": embedded_count,
            "unembedded": total-embedded_count,
            "total_words": sum(doc.word_count() for doc in self._documents.values())
            }
    # ------- DUNDER METHODS ---------

    def __len__(self)-> int:
        return len(self._documents)
    
    def __contains__(self, doc_id:str)-> bool:
        return doc_id in self._documents
    
    def __iter__(self) -> Iterator[Document]:
        return iter(self._documents.values())
    
    def __str__ (self) -> str:
        return f"DocumentCollection(user= {self.user_id}, docs={len(self)})"
    
# test it
print("=== Building Collection ===\n")
collection = DocumentCollection("jay_123", "legal_docs")

# Add Documents
d1 = Document("RAG combines retrieval and generation", "paper.pdf")
d2 = Document("Vector databases store embeddings efficiently", "paper.pdf")
d3 = Document("Fine tuning adapts pretrained models to specific domains", "blog.pdf")
d4 = Document("LoRA reduces trainable parameters by 90 percent", "blog.pdf")

collection.add(d1)
collection.add(d2)
collection.add(d3)
collection.add(d4)

# Test duplicate
print("\n=== Duplicate Test ====")
collection.add(d1)

#Test len and contains
print(f"=== Collection Info ===")
print(f"Total docs: {len(collection)}")
print(f"d1 in collection: {d1.doc_id in collection}")

# Simulating embedding d1 and d2
d1.embedding = [0.1, 0.2, 0.3]
d2.embedding = [0.4, 0.5, 0.6]

# stats
print(f"=== stats ===")
print(collection.stats())

# Search 
print(f"\n=== Search RAg ===")
results = collection.search("RAG")
for doc in results:
    print(f"Found: {doc.preview()}")

#Unembedded
print(f"\n=== Unembedded Docs ===")
for doc in collection.unembedded():
    print(f"Needs embedding: {doc.preview()}")

#Iterate
print(f"\n === Iterating Collection ===")
for doc in collection:
    print(f"   {doc}")

=== Building Collection ===

[ADDED] b49dd2942685 - RAG combines retrieval and generation
[ADDED] 50ad37fe329d - Vector databases store embeddings efficiently
[ADDED] be420e9b64b2 - Fine tuning adapts pretrained models to specific d...
[ADDED] af40cdc86a41 - LoRA reduces trainable parameters by 90 percent

=== Duplicate Test ====
[WARN] Document b49dd2942685 already exists - skipping
=== Collection Info ===
Total docs: 4
d1 in collection: True
=== stats ===
{'user_id': 'jay_123', 'collection': 'legal_docs', 'total_documents': 4, 'embedded': 2, 'unembedded': 2, 'total_words': 25}

=== Search RAg ===
Found: RAG combines retrieval and generation

=== Unembedded Docs ===
Needs embedding: Fine tuning adapts pretrained models to specific d...
Needs embedding: LoRA reduces trainable parameters by 90 percent

 === Iterating Collection ===
   Document(id=b49dd2942685, source=paper.pdf, words=5)
   Document(id=50ad37fe329d, source=paper.pdf, words=5)
   Document(id=be420e9b64b2, source=blog.pdf,

In [25]:
class PDFDocument(Document):
    """
    A Document subclass specifically for PDF files.
    Adds PDF-specific metadata like page number and total pages.
    In Week 4 PyMuPDF will create these automatically.
    """

    def __init__(
            self,
            text:str,
            source:str,
            page_number:int,
            total_pages:int,
            doc_id: Optional[str] = None,
            metadata: Optional[Dict]=None
    ):
        #Call Parent __init__ first - always
        super().__init__(text, source, doc_id, metadata)

        #Validate PDF specific fields
        if page_number <1:
            raise ValueError("Page number must be atleast 1")
        if page_number> total_pages:
            raise ValueError("Page number cannot exceed total pages")
        
        self.page_number = page_number
        self.total_pages = total_pages

        # Auto add page info to metadata
        self.metadata["page"]= str(page_number)
        self.metadata["total_pages"]=str(total_pages)

    def page_progress(self) -> str:
        """show reading progress through the PDF"""
        percentage = (self.page_number/self.total_pages)*100
        return f"Page {self.page_number}/{self.total_pages} ({percentage:.1f}%)"
    
    def is_first_page(self)-> bool:
        return self.page_number ==1
    
    def is_last_page(self)-> bool:
        return self.page_number == self.total_pages
    
    def __str__(self) -> str:
        """Override parent __str__ to include page info"""
        return (f"PDFDocument (id={self.doc_id}, "
                f"source={self.source}, "
                f"page={self.page_number}/{self.total_pages})")

# Test inheritance
print("=== Creating PDF Documents ===\n")

pdf1 = PDFDocument(
    text="Introduction to RAG systems and their applications",
    source="rag_paper.pdf",
    page_number=1,
    total_pages=24
)

pdf2 = PDFDocument(
    text="Vector databases are the backbone of modern RAG pipelines",
    source="rag_paper.pdf",
    page_number=12,
    total_pages=24
)

pdf3 = PDFDocument(
    text="Conclusion and future directions for RAG research",
    source="rag_paper.pdf",
    page_number=24,
    total_pages=24
)

#Test pdf specific methods
print(f"PDF1: {pdf1}")
print(f"PDF2: {pdf2}")
print(f"PDF3: {pdf3}")

print(f"\nProgress: ")
print(f"   pdf1: {pdf1.page_progress()}")
print(f"   pdf2: {pdf2.page_progress()}")
print(f"   pdf3: {pdf3.page_progress()}")

print(f"\nFirst/Last page checks:")
print(f"   pdf1 is first page: {pdf1.is_first_page()}")
print(f"   pdf3 is first page: {pdf3.is_first_page()}")
print(f"   pdf2 is first page: {pdf2.is_first_page()}")

#Inherited methods still work
print(f"\nInherited methods:")
print(f"  pdf1 word count: {pdf1.word_count()}")
print(f"  'RAG' in pdf2: {'RAG' in pdf2}")
print(f"  pdf1 metadata: {pdf1.metadata}")

#isinstance checks
print(f"\n Type checks: ")
print(f"   pdf1 is document: {isinstance(pdf1, Document)}")
print(f"   pdf1 is PDFDocument: {isinstance(pdf1, PDFDocument)}")
print(f"   d1 is PDFDocument: {isinstance(d1, PDFDocument)}")

#Test Validation
print(f"\n=== Validation ===")
try:
    bad = PDFDocument("some text", "file.pdf", page_number=0, total_pages=10)
except ValueError as e:
    print(f"Caught: {e}")

try:
    bad = PDFDocument("some text", "file.pdf", page_number=15, total_pages=10)
except ValueError as e:
    print(f"Caught: {e}" )



=== Creating PDF Documents ===

PDF1: PDFDocument (id=885b2a2e27f4, source=rag_paper.pdf, page=1/24)
PDF2: PDFDocument (id=b3a43a233ef1, source=rag_paper.pdf, page=12/24)
PDF3: PDFDocument (id=85d307f3713b, source=rag_paper.pdf, page=24/24)

Progress: 
   pdf1: Page 1/24 (4.2%)
   pdf2: Page 12/24 (50.0%)
   pdf3: Page 24/24 (100.0%)

First/Last page checks:
   pdf1 is first page: True
   pdf3 is first page: False
   pdf2 is first page: False

Inherited methods:
  pdf1 word count: 7
  'RAG' in pdf2: True
  pdf1 metadata: {'page': '1', 'total_pages': '24'}

 Type checks: 
   pdf1 is document: True
   pdf1 is PDFDocument: True
   d1 is PDFDocument: False

=== Validation ===
Caught: Page number must be atleast 1
Caught: Page number cannot exceed total pages
